# 06 Gold DLT Pipeline — Datamart Layer

**Laag:** Gold / Datamart  
**Schema:** `DATAMART`  
**Pipeline type:** Lakeflow Declarative Pipeline (DLT)

## Wat doet deze pipeline?

Leest van `INTEGRATION.order_header` (order-grain) en berekent de eerste
Gold Materialised View: `DATAMART.daily_sales_by_truck`.  Zie `CONTEXT.md §8`.

**Bron-keuze:** aggregaten lezen van `order_header` (order-grain) om
`SUM`-over-duplicated-line-rijen te voorkomen.  Detail-rijen (meerdere per order)
zouden `order_total`, `order_tax_amount` en `order_discount_amount` meervoudig
optellen als we van `sales_line` zouden aggregeren.

**NULL truck_id:** `truck_id IS NOT NULL` is een `warn`-regel in Silver — rijen
met een ontbrekende truck_id passeren de clean-filter en landen in `order_header`.
Ze verschijnen hier als één `truck_id = NULL`-rij in het aggregaat (het "Unknown"
bucket).  Dit is bewust — zie `CONTEXT.md §8`.

## Tabellen in deze pipeline (slice #22)

| Tabel | Type | Grain | Inhoud |
|---|---|---|---|
| `DATAMART.daily_sales_by_truck` | Materialised View | (order_date, truck_id) | KPI: revenue per truck per dag |


In [ ]:
import dlt
from pyspark.sql import functions as F

## Parameters

DLT reads pipeline parameters from `spark.conf`.  The `catalog` parameter is passed
by the DAB pipeline configuration (`resources/dlt_datamart.yml`).

In [ ]:
catalog = spark.conf.get("pipeline.catalog", "DEMO")
silver_schema = "INTEGRATION"

print(f"Catalog : {catalog}")
print(f"Silver  : {catalog}.{silver_schema}")

## daily_sales_by_truck — Materialised View

Aggregates `INTEGRATION.order_header` at `(order_date, truck_id)` grain.

**Column spec** (CONTEXT.md §8):

| Column | Type | Computation |
|---|---|---|
| `order_date` | DATE | `CAST(order_ts AS DATE)` |
| `truck_id` | DECIMAL(38, 0) | group key |
| `total_orders` | BIGINT | `COUNT(*)` |
| `total_revenue` | DECIMAL(38, 4) | `SUM(order_total)` |
| `total_tax` | DECIMAL(38, 4) | `SUM(order_tax_amount)` |
| `total_discount` | DECIMAL(38, 4) | `SUM(order_discount_amount)` |
| `avg_order_value` | DECIMAL(38, 4) | `total_revenue / total_orders` |

**Pattern:** `@dlt.table` using `spark.read` (batch semantics) = Materialised View.
DLT fully recomputes this on every pipeline run; Silver changes (corrections,
late arrivals) propagate automatically on the next refresh.

**NULL truck_id rows:** Rows where `truck_id IS NULL` (orphan-truck orders that
passed Silver's warn rule) are intentionally kept.  They surface as a single
`truck_id = NULL` bucket — visible data-attribution issues for analysts
(CONTEXT.md §8).

In [ ]:
@dlt.table(
    name="daily_sales_by_truck",
    comment=(
        "KPI aggregate: daily revenue per truck from INTEGRATION.order_header. "
        "Grain: (order_date, truck_id). NULL truck_id rows kept as Unknown bucket "
        "(CONTEXT.md §8). Materialised View — fully recomputed on each pipeline run."
    ),
)
def daily_sales_by_truck():
    """Materialised View — daily revenue KPI aggregated per truck.

    Reads INTEGRATION.order_header via spark.read (batch semantics).  DLT treats
    a @dlt.table that uses spark.read as a Materialised View: it is fully
    recomputed on every pipeline run, so Silver corrections propagate automatically.

    Grain: one row per (order_date, truck_id) combination.  NULL truck_id values
    are intentionally retained — they appear as a single Unknown bucket that
    surfaces orphan-truck orders for analyst triage (CONTEXT.md §8).

    avg_order_value is cast to DECIMAL(38,4) after division to match the column
    spec; the division itself promotes to Double, so an explicit cast is required.
    """
    oh = spark.read.table(f"{catalog}.{silver_schema}.order_header")

    agg = (
        oh
        .withColumn("order_date", F.col("order_ts").cast("date"))
        .groupBy("order_date", "truck_id")
        .agg(
            F.count("*")                        .cast("bigint")         .alias("total_orders"),
            F.sum("order_total")                .cast("decimal(38,4)") .alias("total_revenue"),
            F.sum("order_tax_amount")           .cast("decimal(38,4)") .alias("total_tax"),
            F.sum("order_discount_amount")      .cast("decimal(38,4)") .alias("total_discount"),
        )
    )

    return agg.withColumn(
        "avg_order_value",
        (F.col("total_revenue") / F.col("total_orders")).cast("decimal(38,4)"),
    ).select(
        "order_date",
        "truck_id",
        "total_orders",
        "total_revenue",
        "total_tax",
        "total_discount",
        "avg_order_value",
    )

## Demo notes

- **DLT graph view**: This pipeline is registered as a separate DLT pipeline
  (`resources/dlt_datamart.yml`, target schema `DATAMART`).  The Databricks UI
  shows it as a distinct graph from the Silver pipeline — one node:
  `daily_sales_by_truck` reading from `INTEGRATION.order_header`.
- **NULL truck_id demo**: Insert a Bronze row with `TRUCK_ID = NULL` (or omit it).
  After the end-to-end workflow run, `DATAMART.daily_sales_by_truck` will contain
  an aggregate row where `truck_id IS NULL` with the correct order count and revenue.
- **MV propagation demo**: Correct an `order_total` in `STAGING_AZURESTORAGE` and
  rerun the workflow.  Silver propagates the correction via CDF + apply_changes;
  the next Gold pipeline run fully recomputes `daily_sales_by_truck`, picking up
  the updated aggregate.
- **Workflow chain**: `setup → ingest_* → dlt_integration → dlt_datamart`.
  The `dlt_datamart` task depends on `dlt_integration` so Silver is always
  up-to-date before the Gold aggregation runs.
